# 29 — Available 4B training-data audit (CPU only)

Choose **Runtime → Change runtime type → Hardware accelerator: None**.
Then **Run all** and allow Google Drive.
Upload this notebook directly. No Git push or repository clone is needed.

The original 2B training set and its saved answers/grades are unavailable.
We have stopped that recovery attempt. This audit uses the existing **73-example 4B set**.
It is evidence about 4B training examples, not the old 2B adapter.
The saved 4B training-log summary is frozen here; no separate loss file is needed on Drive.

**Time estimate:** 5–15 minutes, mostly setup/download. Not measured yet.
Use a CPU runtime; the first cell stops if a GPU is connected.
No model weights, generation, training, or generated-code execution are needed.

**Question:** does this available set contain enough repeated drafts to study?
**Hypothesis:** some repeated drafts receive avoidable repeated supervision.
**Change:** compute candidate loss weights only; no weight updates.
**Measure:** duplicate token share and mathematical weight checks.
**Compare:** ordinary token modifiers and a phase-balanced objective.
**Data:** 4B saved training examples. Use the 464 test IDs only for separation checks.
**Metric:** redundant-token-equivalent share of thinking tokens.
**Support for further review:** at least 5%, source checks passing, and useful repetitions inspected.
**Contradiction:** below 5%, wrong sources, overlap, or failed checks.
Passing is a data screen, not proof of efficacy or authorization to train a 4B model.

Outputs: report.json and audit.jsonl, saved to a new Drive folder and downloaded as a ZIP.
No Drive search or file-selection prompt. Missing/wrong input produces a saved blocked report.

**Parser correction:** this version includes indented draft fences. The earlier 1.59%
report skipped these blocks and cannot decide the data screen. Rerun CPU only.


## 1. Check CPU and set the input paths

**Problem/why:** avoid GPU spending and auditing the wrong examples.
**Input:** runtime and Drive paths. **Output:** a new report folder.
**How:** read existing files; write only new audit outputs.


In [ ]:
import os, sys, subprocess, time, json, hashlib, uuid, zipfile, datetime
from pathlib import Path

probe = subprocess.run(['bash','-lc','command -v nvidia-smi >/dev/null && nvidia-smi -L || true'],
                       capture_output=True,text=True,timeout=15)
if 'GPU ' in probe.stdout or os.environ.get('COLAB_GPU') == '1':
    raise RuntimeError('Choose Hardware accelerator: None, reconnect, then Run all. Do not use GPU units for this audit.')
from google.colab import drive, files
drive.mount('/content/drive')
START = time.monotonic()
BASE = Path('/content/drive/MyDrive/stop-overthinking')
TRAIN_FILE = BASE/'results/4b/raw/train-set-lora1.jsonl'
# This available 4B file is a new audit source, not a substitute for historical 2B data.
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
OUT = BASE/'shared-draft-audit'/RUN_ID
OUT.mkdir(parents=True,exist_ok=False)
report = {'run_id':RUN_ID,'status':'started','scope':'CPU data and loss-weight audit; no efficacy evidence',
          'gpu_used':False, 'audit_dataset':'4B selected training traces', 'train_file':str(TRAIN_FILE)}
records=[]
def save_report():
    report['wall_seconds']=round(time.monotonic()-START,3)
    temp=OUT/'report.tmp'
    temp.write_text(json.dumps(report,indent=2))
    temp.replace(OUT/'report.json')
def download_report():
    save_report()
    archive=Path('/content')/(RUN_ID+'-draft-audit.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for path in OUT.iterdir():
            if path.is_file(): z.write(path,path.name)
    print(json.dumps(report,indent=2))
    print('Saved:',OUT)
    files.download(str(archive))
save_report()
print('CPU audit only. Input:',TRAIN_FILE)


## 2. Load the tokenizer and audit the data

**Problem/why:** repeated blocks may occupy very few training tokens.
**Input:** available 4B training examples and a pinned tokenizer (text splitter).
**Output:** token shares, source checks, and candidate-weight checks.
**How:** parse Python structure without executing code. Other languages stay supervised.
No attention matrices or model weights are loaded.

The answer's **raw repetition modifier is one**. Its mean-loss coefficient is also one.
Per-token coefficients sum to one within the answer term, so they need not each equal one.
No-duplicate modifiers match ordinary weighting. The full group objective also changes
phase/length balancing, which needs a separate matched comparison in a later experiment.

The saved 73-example source must match its frozen export hash.
Historical tokenizer identity and semantic test separation remain unproved.


In [ ]:
SPEC = {'evaluation_ids': ['HumanEval/0', 'HumanEval/1', 'HumanEval/10', 'HumanEval/100', 'HumanEval/101', 'HumanEval/102', 'HumanEval/103', 'HumanEval/104', 'HumanEval/105', 'HumanEval/106', 'HumanEval/107', 'HumanEval/108', 'HumanEval/109', 'HumanEval/11', 'HumanEval/110', 'HumanEval/111', 'HumanEval/112', 'HumanEval/113', 'HumanEval/114', 'HumanEval/115', 'HumanEval/116', 'HumanEval/117', 'HumanEval/118', 'HumanEval/119', 'HumanEval/12', 'HumanEval/120', 'HumanEval/121', 'HumanEval/122', 'HumanEval/123', 'HumanEval/124', 'HumanEval/125', 'HumanEval/126', 'HumanEval/127', 'HumanEval/128', 'HumanEval/129', 'HumanEval/13', 'HumanEval/130', 'HumanEval/131', 'HumanEval/132', 'HumanEval/133', 'HumanEval/134', 'HumanEval/135', 'HumanEval/136', 'HumanEval/137', 'HumanEval/138', 'HumanEval/139', 'HumanEval/14', 'HumanEval/140', 'HumanEval/141', 'HumanEval/142', 'HumanEval/143', 'HumanEval/144', 'HumanEval/145', 'HumanEval/146', 'HumanEval/147', 'HumanEval/148', 'HumanEval/149', 'HumanEval/15', 'HumanEval/150', 'HumanEval/151', 'HumanEval/152', 'HumanEval/153', 'HumanEval/154', 'HumanEval/155', 'HumanEval/156', 'HumanEval/157', 'HumanEval/158', 'HumanEval/159', 'HumanEval/16', 'HumanEval/160', 'HumanEval/161', 'HumanEval/162', 'HumanEval/163', 'HumanEval/17', 'HumanEval/18', 'HumanEval/19', 'HumanEval/2', 'HumanEval/20', 'HumanEval/21', 'HumanEval/22', 'HumanEval/23', 'HumanEval/24', 'HumanEval/25', 'HumanEval/26', 'HumanEval/27', 'HumanEval/28', 'HumanEval/29', 'HumanEval/3', 'HumanEval/30', 'HumanEval/31', 'HumanEval/32', 'HumanEval/33', 'HumanEval/34', 'HumanEval/35', 'HumanEval/36', 'HumanEval/37', 'HumanEval/38', 'HumanEval/39', 'HumanEval/4', 'HumanEval/40', 'HumanEval/41', 'HumanEval/42', 'HumanEval/43', 'HumanEval/44', 'HumanEval/45', 'HumanEval/46', 'HumanEval/47', 'HumanEval/48', 'HumanEval/49', 'HumanEval/5', 'HumanEval/50', 'HumanEval/51', 'HumanEval/52', 'HumanEval/53', 'HumanEval/54', 'HumanEval/55', 'HumanEval/56', 'HumanEval/57', 'HumanEval/58', 'HumanEval/59', 'HumanEval/6', 'HumanEval/60', 'HumanEval/61', 'HumanEval/62', 'HumanEval/63', 'HumanEval/64', 'HumanEval/65', 'HumanEval/66', 'HumanEval/67', 'HumanEval/68', 'HumanEval/69', 'HumanEval/7', 'HumanEval/70', 'HumanEval/71', 'HumanEval/72', 'HumanEval/73', 'HumanEval/74', 'HumanEval/75', 'HumanEval/76', 'HumanEval/77', 'HumanEval/78', 'HumanEval/79', 'HumanEval/8', 'HumanEval/80', 'HumanEval/81', 'HumanEval/82', 'HumanEval/83', 'HumanEval/84', 'HumanEval/85', 'HumanEval/86', 'HumanEval/87', 'HumanEval/88', 'HumanEval/89', 'HumanEval/9', 'HumanEval/90', 'HumanEval/91', 'HumanEval/92', 'HumanEval/93', 'HumanEval/94', 'HumanEval/95', 'HumanEval/96', 'HumanEval/97', 'HumanEval/98', 'HumanEval/99', 'lcb/3026', 'lcb/3328', 'lcb/3334', 'lcb/3337', 'lcb/3346', 'lcb/3349', 'lcb/3353', 'lcb/3354', 'lcb/3355', 'lcb/3356', 'lcb/3360', 'lcb/3361', 'lcb/3367', 'lcb/3371', 'lcb/3372', 'lcb/3374', 'lcb/3379', 'lcb/3381', 'lcb/3384', 'lcb/3394', 'lcb/3395', 'lcb/3396', 'lcb/3397', 'lcb/3398', 'lcb/3403', 'lcb/3406', 'lcb/3408', 'lcb/3412', 'lcb/3421', 'lcb/3422', 'lcb/3426', 'lcb/3428', 'lcb/3429', 'lcb/3442', 'lcb/3446', 'lcb/3447', 'lcb/3450', 'lcb/3451', 'lcb/3453', 'lcb/3455', 'lcb/3456', 'lcb/3462', 'lcb/3463', 'lcb/3464', 'lcb/3468', 'lcb/3469', 'lcb/3471', 'lcb/3475', 'lcb/3476', 'lcb/3479', 'lcb/3484', 'lcb/3487', 'lcb/3493', 'lcb/3494', 'lcb/3496', 'lcb/3498', 'lcb/3502', 'lcb/3507', 'lcb/3508', 'lcb/3511', 'lcb/3515', 'lcb/3517', 'lcb/3518', 'lcb/3519', 'lcb/3522', 'lcb/3526', 'lcb/3528', 'lcb/3533', 'lcb/3534', 'lcb/3540', 'lcb/3543', 'lcb/3553', 'lcb/3555', 'lcb/3558', 'lcb/3559', 'lcb/3567', 'lcb/3568', 'lcb/3570', 'lcb/3573', 'lcb/3579', 'lcb/3587', 'lcb/3591', 'lcb/3593', 'lcb/3600', 'lcb/3605', 'lcb/3606', 'lcb/3607', 'lcb/3610', 'lcb/3612', 'lcb/3613', 'lcb/3616', 'lcb/3617', 'lcb/3620', 'lcb/3621', 'lcb/3622', 'lcb/3625', 'lcb/3626', 'lcb/3627', 'lcb/3629', 'lcb/3632', 'lcb/3634', 'lcb/3636', 'lcb/3639', 'lcb/3644', 'lcb/3649', 'lcb/3651', 'lcb/3653', 'lcb/3654', 'lcb/3655', 'lcb/3656', 'lcb/3657', 'lcb/3659', 'lcb/3676', 'lcb/3681', 'lcb/3683', 'lcb/3684', 'lcb/3685', 'lcb/3686', 'lcb/3699', 'lcb/3702', 'lcb/3705', 'lcb/3709', 'lcb/3715', 'lcb/3716', 'lcb/3722', 'lcb/3723', 'lcb/3736', 'lcb/3743', 'lcb/3748', 'lcb/3750', 'lcb/3753', 'lcb/3754', 'lcb/3759', 'lcb/3760', 'lcb/3763', 'lcb/3764', 'lcb/3768', 'lcb/3771', 'lcb/3773', 'lcb/3776', 'lcb/3778', 'lcb/3779', 'lcb/3785', 'lcb/3786', 'lcb/3788', 'lcb/3791', 'lcb/3793', 'lcb/3794', 'lcb/3795', 'lcb/3799', 'lcb/3805', 'lcb/3809', 'lcb/3811', 'lcb/3817', 'lcb/3832', 'lcb/abc344_a', 'lcb/abc344_b', 'lcb/abc345_b', 'lcb/abc346_a', 'lcb/abc346_b', 'lcb/abc347_a', 'lcb/abc347_b', 'lcb/abc348_a', 'lcb/abc348_b', 'lcb/abc349_a', 'lcb/abc349_b', 'lcb/abc349_c', 'lcb/abc350_a', 'lcb/abc350_b', 'lcb/abc351_a', 'lcb/abc351_b', 'lcb/abc352_a', 'lcb/abc353_a', 'lcb/abc353_b', 'lcb/abc354_a', 'lcb/abc354_b', 'lcb/abc355_a', 'lcb/abc355_b', 'lcb/abc356_a', 'lcb/abc356_b', 'lcb/abc356_c', 'lcb/abc356_d', 'lcb/abc357_a', 'lcb/abc357_b', 'lcb/abc357_c', 'lcb/abc357_d', 'lcb/abc358_a', 'lcb/abc358_b', 'lcb/abc358_c', 'lcb/abc359_a', 'lcb/abc359_b', 'lcb/abc359_c', 'lcb/abc360_a', 'lcb/abc360_b', 'lcb/abc361_a', 'lcb/abc361_b', 'lcb/abc362_a', 'lcb/abc362_b', 'lcb/abc363_a', 'lcb/abc363_b', 'lcb/abc363_c', 'lcb/abc363_d', 'lcb/abc364_a', 'lcb/abc364_b', 'lcb/abc365_a', 'lcb/abc365_b', 'lcb/abc366_a', 'lcb/abc366_b', 'lcb/abc367_a', 'lcb/abc367_b', 'lcb/abc368_a', 'lcb/abc368_b', 'lcb/abc369_a', 'lcb/abc369_b', 'lcb/abc370_a', 'lcb/abc370_b', 'lcb/abc370_c', 'lcb/abc371_a', 'lcb/abc371_b', 'lcb/abc371_c', 'lcb/abc372_a', 'lcb/abc373_a', 'lcb/abc373_b', 'lcb/abc374_a', 'lcb/abc374_b', 'lcb/abc374_c', 'lcb/abc374_d', 'lcb/abc375_a', 'lcb/abc375_d', 'lcb/abc376_a', 'lcb/abc376_b', 'lcb/abc377_a', 'lcb/abc377_b', 'lcb/abc378_a', 'lcb/abc378_b', 'lcb/abc379_a', 'lcb/abc379_b', 'lcb/abc379_d', 'lcb/abc380_a', 'lcb/abc380_b', 'lcb/abc380_c', 'lcb/abc380_d', 'lcb/abc381_a', 'lcb/abc381_b', 'lcb/abc381_c', 'lcb/abc382_a', 'lcb/abc382_b', 'lcb/abc382_c', 'lcb/abc383_a', 'lcb/abc383_b', 'lcb/abc383_c', 'lcb/abc383_d', 'lcb/abc384_a', 'lcb/abc384_b', 'lcb/abc384_c', 'lcb/abc384_d', 'lcb/abc385_a', 'lcb/abc385_b', 'lcb/abc385_c', 'lcb/abc386_a', 'lcb/abc386_b', 'lcb/abc386_c', 'lcb/abc391_a', 'lcb/abc391_b', 'lcb/abc391_d', 'lcb/abc392_a', 'lcb/abc392_b', 'lcb/abc392_c', 'lcb/abc392_d', 'lcb/abc393_a', 'lcb/abc393_b', 'lcb/abc394_a', 'lcb/abc394_b', 'lcb/abc394_c', 'lcb/abc394_d', 'lcb/abc395_a', 'lcb/abc395_b', 'lcb/abc395_c', 'lcb/abc396_a', 'lcb/abc396_b', 'lcb/abc396_c', 'lcb/abc396_d', 'lcb/abc397_a', 'lcb/abc397_b', 'lcb/abc397_c', 'lcb/abc398_a', 'lcb/abc398_b', 'lcb/abc398_c', 'lcb/abc399_a', 'lcb/abc399_b', 'lcb/abc399_c', 'lcb/abc399_d', 'lcb/abc400_a', 'lcb/abc400_b', 'lcb/abc400_c', 'lcb/abc400_d', 'lcb/arc189_a', 'lcb/arc192_a', 'lcb/arc194_a', 'lcb/arc195_a'], 'loss_sha256': '7b3cc6e4f37220464c2964087ea6787b74b5858588fa1aa7a3e8c982b3dce523', 'expected_examples': 73, 'expected_epochs': 3, 'expected_fraction': 1.0, 'tokenizer': 'Qwen/Qwen3.5-4B', 'tokenizer_revision': '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a', 'expected_train_sha256': 'e2e2b7b8f87ad33a553dbd94dedc654728924227f3dc42ac6ed83ab55e7ec537', 'training_log': {'n_examples': 73, 'epochs': 3, 'fraction': 1.0, 'source': 'results/4b/raw/lora/lora1/loss.json'}}

import ast, collections, math, re, textwrap

# Only Python or unlabelled fenced blocks qualify. Other languages stay supervised.
FENCE = re.compile(r'^(?P<indent>[ \t]*)```(?P<language>[^\n`]*)\n(?P<body>.*?)^(?P=indent)```[ \t]*$', re.M | re.S)

def analyse(completion, offsets):
    close = completion.find('</think>')
    if close < 0:
        raise ValueError('Missing thinking closure: do not silently treat a cut-off trace as a final answer.')
    groups, invalid, excluded = collections.defaultdict(list), 0, 0
    for match in FENCE.finditer(completion[:close]):
        language = match.group('language').strip().lower()
        if language not in ('', 'python', 'py'):
            excluded += 1
            continue
        try:
            tree = ast.parse(textwrap.dedent(match.group('body')))
        except (SyntaxError, ValueError):
            invalid += 1
            continue
        if not tree.body:
            continue
        key = ast.dump(tree, include_attributes=False)
        groups[key].append(match.span('body'))
    # A token crossing a code boundary stays ordinary reasoning. Never guess its span.
    tokens = [[[] for _ in spans] for spans in groups.values()]
    categories = ['prose'] * len(offsets)
    for i, (a, b) in enumerate(offsets):
        if a >= close:
            categories[i] = 'answer'
        elif b > close:
            categories[i] = 'answer'  # boundary token remains protected
        if b <= a:
            continue  # EOS is assigned explicitly by the caller
        for gi, spans in enumerate(groups.values()):
            for bi, (lo, hi) in enumerate(spans):
                if lo <= a < b <= hi:
                    categories[i] = 'code'
                    tokens[gi][bi].append(i)
    # Exact group objective from the note. Empty token blocks fall back to prose.
    active = [[ids for ids in group if ids] for group in tokens]
    active = [group for group in active if group]
    grouped_ids = {i for group in active for ids in group for i in ids}
    for i, category in enumerate(categories):
        if category == 'code' and i not in grouped_ids:
            categories[i] = 'prose'
    weights = [0.0] * len(offsets)
    balanced = [0.0] * len(offsets)
    # lambda=mu=1 fixed for this mathematical audit, not tuned on test data.
    for category in ('prose', 'answer'):
        ids = [i for i, c in enumerate(categories) if c == category]
        for i in ids:
            weights[i] = balanced[i] = 1 / len(ids)
    code_ids = sorted(grouped_ids)
    for i in code_ids:
        balanced[i] = 1 / len(code_ids)
    for group in active:
        for ids in group:
            for i in ids:
                weights[i] = 1 / (len(active) * len(group) * len(ids))
    modifiers = [1.0] * len(offsets)
    redundant_equivalent = 0.0
    duplicate_tokens = 0
    for group in active:
        for ids in group:
            for i in ids:
                modifiers[i] = 1 / len(group)
        if len(group) > 1:
            n = sum(map(len, group))
            duplicate_tokens += n
            redundant_equivalent += n * (1 - 1 / len(group))
    return dict(weights=weights, balanced=balanced, modifiers=modifiers,
        categories=categories, groups=active, missing_closure=False,
        thinking_tokens=sum(c != 'answer' for c in categories),
        answer_tokens=categories.count('answer'), total_tokens=len(offsets),
        duplicate_group_tokens=duplicate_tokens,
        redundant_token_equivalent=redundant_equivalent,
        duplicate_groups=sum(len(g)>1 for g in active),
        parseable_blocks=sum(map(len,tokens)), invalid_blocks=invalid,
        excluded_language_blocks=excluded,
        boundary_tokens=sum(a < close < b for a,b in offsets))

def shifted_labels(prompt_ids, completion_ids, weights):
    ids = prompt_ids + completion_ids
    labels = [-100] * len(prompt_ids) + completion_ids
    target_weights = [0.0] * len(prompt_ids) + weights
    # logits at position t predict label t+1. Shift both labels and weights together.
    return ids, labels[1:], target_weights[1:]

def mathematical_checks():
    def fixture(text):
        return analyse(text, [(i,i+1) for i in range(len(text))])
    a = '```python\nx = 1\n```\n'
    b = '```python\nx = 2\n```\n'
    one = fixture(a + '</think>\nfinal')
    twice = fixture(a+a+'</think>\nfinal')
    changed = fixture(a+b+'</think>\nfinal')
    assert one['duplicate_groups'] == 0 and all(v==1 for v in one['modifiers'])
    assert twice['duplicate_groups'] == 1 and changed['duplicate_groups'] == 0
    for result in (one,twice,changed):
        assert all(math.isfinite(w) and w>=0 for w in result['weights'])
        answer = [i for i,c in enumerate(result['categories']) if c=='answer']
        assert all(result['modifiers'][i]==1 for i in answer)
        assert math.isclose(sum(result['weights'][i] for i in answer),1)
        assert all(result['weights'][i]==result['balanced'][i] for i in answer)
        assert math.isclose(sum(result['weights'][i] for g in result['groups'] for block in g for i in block),1)
    ids, labels, weights = shifted_labels([10,11],[20,21,22],[.2,.3,.5])
    assert labels==[-100,20,21,22] and weights==[0,.2,.3,.5]
    # Cross entropy depends on context; test budget invariance, not gradient invariance.
    assert math.isclose(sum(w*2 for i,w in enumerate(one['weights']) if one['categories'][i]=='code'),2)
    assert math.isclose(sum(w*2 for i,w in enumerate(twice['weights']) if twice['categories'][i]=='code'),2)
    for indent in ('    ', '        ', '\t'):
        indented=''.join(indent+line+'\n' for line in a.splitlines())
        result=fixture(indented+indented+'</think>final')
        assert result['parseable_blocks']==2 and result['duplicate_groups']==1
    other = fixture('```javascript\nx = 1;\n```\n</think>final')
    assert other['excluded_language_blocks']==1 and not other['groups']
    # No-duplicate modifiers are identity. The full group objective differs from token CE.
    return ['code revision retained', 'duplicate group budget invariant',
            'answer modifier one and answer loss budget one', 'shifted prompt masking',
            'finite nonnegative weights', 'indented draft blocks retained', 'non-Python excluded', 'no-duplicate modifier identity']

try:
    missing=[str(TRAIN_FILE)] if not TRAIN_FILE.is_file() else []
    if missing:
        report.update(status='blocked_missing_source',missing_files=missing,
                      next_step='Expected available 4B training file is missing. Check the exact saved Drive path; no model generation or training.')
    else:
        data=TRAIN_FILE.read_bytes()
        rows=[json.loads(line) for line in data.decode('utf-8').splitlines() if line.strip()]
        log=SPEC['training_log']
        report['train_sha256']=hashlib.sha256(data).hexdigest()
        report['saved_training_log']={**SPEC['training_log'],'sha256':SPEC['loss_sha256']}
        report['historical_tokenizer_identity']='Unverified: counts use the pinned official tokenizer, not an archived Unsloth tokenizer.'
        ids=[r['task_id'] for r in rows]
        overlap=sorted(set(ids)&set(SPEC['evaluation_ids']))
        source_checks={
          'exported_training_file_hash_matches':report['train_sha256']==SPEC['expected_train_sha256'],
          'example_count_matches':len(rows)==SPEC['expected_examples']==log.get('n_examples'),
          'training_settings_match':log.get('epochs')==SPEC['expected_epochs'] and log.get('fraction')==SPEC['expected_fraction'],
          'unique_training_ids':len(ids)==len(set(ids)),
          'no_exact_test_id_overlap':not overlap}
        report.update(source_checks=source_checks,test_id_overlap=overlap,
          provenance='Notebook 15b workflow plus frozen saved 4B training log and exact hash of exported 4B training file. No training-time hash: historical identity remains unproved.',
          semantic_overlap_check='Not performed. Exact ID separation alone does not rule out paraphrased questions.')
        if not all(source_checks.values()):
            report.update(status='blocked_source_mismatch',next_step='Resolve source mismatch before considering training.')
        else:
            subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.18.0'],check=True,timeout=300)
            from transformers import AutoTokenizer
            tok=AutoTokenizer.from_pretrained(SPEC['tokenizer'],revision=SPEC['tokenizer_revision'],use_fast=True)
            assert tok.is_fast and tok.eos_token_id is not None
            checks=mathematical_checks()
            for row in rows:
                if time.monotonic()-START > 900:
                    raise TimeoutError('15-minute audit ceiling reached. Partial records are saved.')
                completion=row['completion']
                encoded=tok(completion,add_special_tokens=False,return_offsets_mapping=True)
                comp_ids=encoded['input_ids']+[tok.eos_token_id]
                # Follow scripts/train_lora.py encoding rule under this pinned official tokenizer.
                assert comp_ids==tok(completion+tok.eos_token,add_special_tokens=False)['input_ids']
                offsets=list(encoded['offset_mapping'])+[(len(completion),len(completion)+1)]
                result=analyse(completion,offsets)
                prompt=tok.apply_chat_template([{'role':'user','content':row['question']}],
                  add_generation_prompt=True,tokenize=False,enable_thinking=True)
                prompt_ids=tok(prompt,add_special_tokens=False)['input_ids']
                full, labels, shifted=shifted_labels(prompt_ids,comp_ids,result['weights'])
                assert labels[len(prompt_ids)-1]==comp_ids[0]
                assert shifted[len(prompt_ids)-1]==result['weights'][0]
                assert all(w==0 for y,w in zip(labels,shifted) if y==-100)
                answer=[i for i,c in enumerate(result['categories']) if c=='answer']
                assert answer and all(result['modifiers'][i]==1 for i in answer)
                assert math.isclose(sum(result['weights'][i] for i in answer),1,abs_tol=1e-8)
                n=len(full)
                record={k:v for k,v in result.items() if k not in ('weights','balanced','modifiers','categories','groups')}
                record.update(task_id=row['task_id'],sequence_tokens=n,prompt_tokens=len(prompt_ids),
                    completion_sha256=hashlib.sha256(completion.encode()).hexdigest(),
                    answer_raw_modifier_min=1.0,answer_loss_budget=sum(result['weights'][i] for i in answer),
                    hypothetical_cutoff_4096=n>4096,hypothetical_cutoff_8192=n>8192)
                records.append(record)
                with (OUT/'audit.jsonl').open('a') as f:f.write(json.dumps(record)+'\n')
            total=sum(r['total_tokens'] for r in records)
            think=sum(r['thinking_tokens'] for r in records)
            redundant=sum(r['redundant_token_equivalent'] for r in records)
            share=redundant/think if think else 0.0
            report.update(status='audit_passed',parser_version='indent-aware-v2',examples=len(records),checks=checks+['actual tokenizer EOS identity','actual shifted label alignment'],
              tokenizer=SPEC['tokenizer'],tokenizer_revision=SPEC['tokenizer_revision'],
              duplicate_traces=sum(r['duplicate_groups']>0 for r in records),
              completion_tokens=total,thinking_tokens=think,
              duplicate_group_tokens=sum(r['duplicate_group_tokens'] for r in records),
              redundant_token_equivalent=redundant,
              redundant_share_of_thinking=share,
              redundant_share_of_completion=redundant/total if total else 0.0,
              invalid_blocks=sum(r['invalid_blocks'] for r in records),
              excluded_language_blocks=sum(r['excluded_language_blocks'] for r in records),
              truncation_note='Original script uses the longest full sequence, with no explicit truncation. Cutoff flags are hypothetical only.',
              longest_full_sequence=max(r['sequence_tokens'] for r in records),
              screen_threshold=0.05,screen_passed=share>=0.05,
              next_step=('Review useful versus redundant repetitions and complete provenance/semantic separation before pricing one GPU pilot.'
                         if share>=0.05 else 'Park this candidate: affected thinking supervision is below the preset 5% screen. No GPU training.'))
            print(f"Audited {len(records)} actual training examples. Repeated-draft equivalent: {share:.1%} of thinking tokens.")
except Exception as error:
    report.update(status='failed',error_type=type(error).__name__,error=str(error),completed_examples=len(records),
                  next_step='Inspect saved error; do not start training.')
finally:
    download_report()


## 3. What next?

Send the downloaded ZIP here. It contains the report and per-example audit counts.
Do not start training from this notebook's screen alone.
We must inspect the data, source limits, and useful repetitions first.
Disconnect the CPU runtime when finished. Old notebooks and results remain unchanged.
